# Brent · `postcode` (web: "Postcode")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly.

Brent only (golden rule): every query carries the location-based Brent filter.
Data comes from the **live API**.

Workflow: add a step here → discuss the findings → decide whether it belongs in
`knowledge/brent.md` → add it.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and empty values

Fetch `postcode` for every Brent application (paged with `search_after`, sorted by
`id`, because there are more than 10,000). Then count:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it, or only whitespace.

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "postcode"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "postcode"])
print("Brent applications:", len(df))

Brent applications: 36952


In [3]:
value = df["postcode"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "filled": int((~is_null & ~is_blank).sum()),
})
summary["total"] = summary.sum()
summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)})

,count,share_%
null,32918,89.08
empty or blank,4,0.01
filled,4030,10.91
total,36952,100.00


### Findings · Step 1

Live API, 2026-11-10, 36,952 Brent applications:

| | Count | Share |
|---|---|---|
| null | 32,918 | 89.08% |
| empty or blank | 4 | 0.01% |
| filled | 4,030 | 10.91% |

- **Mostly empty:** only 10.91% of records have a `postcode`.
- **4 are blank** (empty or whitespace only).

## Step 2 · Do the values match the schema type?

Count the Python type of every value in `df`: `str` = JSON string, `int` / `float` =
JSON number, `NoneType` = null.

In [4]:
types = df["postcode"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
display(types.value_counts().to_frame("count"))

not_str = df[~types.isin(["str", "NoneType"])]
print("values that are not strings:", len(not_str))
not_str["postcode"].value_counts().head(30).to_frame("count")

,count
postcode,
NoneType,32918
str,4034


values that are not strings: 0


,count
postcode,


### Findings · Step 2

Live API, 2026-11-10:

| Python type | Count |
|---|---|
| `str` | 4,034 |
| null | 32,918 |
| number | 0 |

- **Every filled value is a JSON string**, as the schema says (`string`). No type mismatch.

## Step 3 · Most common values

Count identical values of `postcode` in `df` (exact text) and show the 30 most common.
`shown_as` uses `repr()` so hidden characters are visible.

In [5]:
print("distinct values:", df["postcode"].nunique(), "| filled:", df["postcode"].notna().sum())
top = df["postcode"].value_counts().head(30).to_frame("count").reset_index()
top["shown_as"] = top["postcode"].map(repr)
top

distinct values: 1804 | filled: 4034


,postcode,count,shown_as
0,NW10,211,'NW10'
1,HA9,111,'HA9'
2,HA0,86,'HA0'
3,NW2,75,'NW2'
4,NW6,61,'NW6'
5,NW9,43,'NW9'
6,HA3,23,'HA3'
7,W10 4AY,19,'W10 4AY'
8,NW10 4AE,14,'NW10 4AE'
9,HA0 1TX,13,'HA0 1TX'


### Findings · Step 3

Live API, 2026-11-10: 4,034 filled, 1,804 distinct.

- **Two forms:** full postcodes (`NW10 4AE`) and postcode districts only (`NW10` 211,
  `HA9` 111, `HA0` 86, `NW2` 75, `NW6` 61, `NW9` 43, `HA3` 23). The districts-only values
  are the most common ones, as on the early-1987 legacy records.
- No case or spacing variants in the top 30.
- `W10 4AY` (19) and `HA1 3UJ` (8) are in districts that are mostly outside Brent; not
  checked.